In [3]:
"""
Chatbot Development with a pre-trained transformer (DialoGPT)
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class DialoGPTChatbot:
    def __init__(self, model_name: str = "microsoft/DialoGPT-medium",
                 max_history_tokens: int = 700):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
        self.model.eval()
        self.max_history_tokens = max_history_tokens
        self.history_ids = None  # running conversation as token ids

    def reset(self):
        """Clear the conversation memory."""
        self.history_ids = None

    def chat(self, user_text: str, max_new_tokens: int = 100,
             temperature: float = 0.7, top_k: int = 50, top_p: float = 0.92) -> str:
        # Encode the user message and append the end-of-sequence token
        new_ids = self.tokenizer.encode(
            user_text + self.tokenizer.eos_token, return_tensors="pt"
        ).to(DEVICE)

        # Append to existing history (this gives the bot context/memory)
        input_ids = new_ids if self.history_ids is None else torch.cat(
            [self.history_ids, new_ids], dim=-1
        )

        # Trim the oldest tokens so we never exceed the context window
        if input_ids.shape[-1] > self.max_history_tokens:
            input_ids = input_ids[:, -self.max_history_tokens:]

        attention_mask = torch.ones_like(input_ids)

        with torch.no_grad():
            output_ids = self.model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                pad_token_id=self.tokenizer.eos_token_id,
                do_sample=True,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                no_repeat_ngram_size=3,
            )

        # Save full conversation (input + reply) as the new history
        self.history_ids = output_ids

        # Decode only the newly generated reply
        reply = self.tokenizer.decode(
            output_ids[0, input_ids.shape[-1]:], skip_special_tokens=True
        )
        return reply.strip() or "I'm not sure what to say to that."

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Initialize the chatbot
print("Loading chatbot model...")
bot = DialoGPTChatbot()
print("Model loaded! Ready to chat.")

# Create widgets for the UI
output_area = widgets.Output()
text_input = widgets.Text(placeholder='Type your message here and press Enter...', layout=widgets.Layout(width='80%'))
clear_button = widgets.Button(description="Reset Conversation", button_style='warning')

# Define interaction logic
def on_submit(sender):
    user_text = text_input.value.strip()
    if not user_text:
        return

    # Clear the text input field
    text_input.value = ''

    with output_area:
        print(f"You: {user_text}")
        # Generate response
        reply = bot.chat(user_text)
        print(f"Bot: {reply}\n" + "-"*50)

def on_clear_click(b):
    bot.reset()
    with output_area:
        clear_output()
        print("System: Conversation memory cleared.\n" + "-"*50)

# Bind handlers
text_input.on_submit(on_submit)
clear_button.on_click(on_clear_click)

# Display the chat widget
display(widgets.VBox([output_area, widgets.HBox([text_input, clear_button])]))

Loading chatbot model...


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Model loaded! Ready to chat.
